# 민정 모델 개선 실험 — v14_lr_C1.0

원본: `../model_민정_v10_original.ipynb` (수정하지 않은 복사본).

설정: `{"version": "v14_lr_C1.0", "alpha": 1.0, "C": 1.0, "mode": "base", "ablate": null}`

train 내부 그룹 3-fold 개발 검증입니다. 분리된 확인용 holdout과 test는 이 노트북에서 평가하거나 읽지 않습니다. 특징 사전·빈도 필터·스케일·IDF·가중치는 각 학습 fold로만 적합합니다. 학습 180회 고정, 중복 라벨 덮어쓰기 없음. 아래 실행 셀은 해당 버전을 재학습합니다.

개발 OOF에서 앙상블 비중을 선택하므로 이 점수에는 선택 편향이 있습니다. 최종 확인 점수는 `summary.json`을 보세요.

저장된 개발 결과:
```json
{
  "version": "v14_lr_C1.0",
  "alpha": "1.0",
  "C": "1.0",
  "mode": "base",
  "ablate": "",
  "xgb_weight": "0.75",
  "cv_macro_f1": "0.4624966361978586",
  "seconds": "2.5",
  "lr_alpha": ""
}
```


In [ ]:
from pathlib import Path
import json
HERE=Path.cwd()
if not (HERE/'run_experiments.py').exists():
    HERE=HERE/'models'/'혜진'/'codex'/'minjung_experiments'
script=HERE/'run_experiments.py'
ns={'__file__':str(script.resolve())}
# Load train-only functions/setup; stop before the full sweep loop.
exec(compile(script.read_text().split('# Query transforms may see')[0],str(script),'exec'),ns)
cfg={'version': 'v14_lr_C1.0', 'alpha': 1.0, 'C': 1.0, 'mode': 'base', 'ablate': None}
np=ns['np']; y=ns['y']; dev=ns['dev']
ox=np.zeros((len(y),ns['n_class'])); ol=ox.copy()
for f,(a,b) in enumerate(ns['folds']):
    px,pl=ns['fit_predict'](a,ns['ex'],ns['docs'],cfg,foldkey=f)
    ox[b]=px[b]; ol[b]=pl[b]
    print('fold completed',f)
from sklearn.metrics import f1_score
scores=[(float(f1_score(y[dev],(w*ox[dev]+(1-w)*ol[dev]).argmax(1),average='macro')),w) for w in [0.,.25,.4,.5,.6,.75,1.]]
score,w=max(scores,key=lambda t:t[0])
print({'version':cfg['version'],'development_macro_f1':score,'xgb_weight':w})
np.savez_compressed(HERE/(cfg['version']+'_oof.npz'),xgb=ox[dev],lr=ol[dev],rows=dev)
